# 🚀 BoneRAG — FracAtlas Multi-Model GPU Indexing (Colab Notebook)

Notebook này cho phép bạn chọn và mã hóa tập dữ liệu **FracAtlas (4,082 ảnh X-quang)** bằng nhiều Foundation Models khác nhau trên Google Colab T4 GPU miễn phí.

### 📌 Hướng dẫn sử dụng:
1. **Bật GPU**: Menu `Runtime` -> `Change runtime type` -> Chọn `T4 GPU`.
2. **Chọn mô hình (MODEL_NAME)** ở Ô [Step 2]:
   - `microsoft/BiomedCLIP-PubMedBERT` (Mặc định cho y khoa)
   - `openai/ViT-B-32` (CLIP tổng quát nhẹ)
   - `openai/ViT-L-14` (CLIP tổng quát cao cấp)
3. Bấm **Runtime -> Run all (Ctrl + F9)**.
4. Tải file `.faiss` và `.json` ở cột thư mục bên trái về máy để dùng trong BoneRAG App!

In [ ]:
# [Step 1] Cài đặt các thư viện cần thiết trên Colab GPU
!pip install -q torch open_clip_torch faiss-cpu pillow tqdm huggingface_hub

In [ ]:
# [Step 2] Chọn Mô hình Foundation Model & Khởi tạo PyTorch
import torch
import open_clip
import faiss
import json
import os
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# @title Chọn mô hình Foundation Model { run: "auto" }
MODEL_CHOICE = "BiomedCLIP (Microsoft)" # @param ["BiomedCLIP (Microsoft)", "OpenAI CLIP ViT-B/32", "OpenAI CLIP ViT-L/14"]

MODEL_MAP = {
    "BiomedCLIP (Microsoft)": {
        "hub": "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
        "prefix": "fracatlas_biomedclip"
    },
    "OpenAI CLIP ViT-B/32": {
        "model": "ViT-B-32",
        "pretrained": "openai",
        "prefix": "fracatlas_clip_vitb32"
    },
    "OpenAI CLIP ViT-L/14": {
        "model": "ViT-L-14",
        "pretrained": "openai",
        "prefix": "fracatlas_clip_vitl14"
    }
}

cfg = MODEL_MAP[MODEL_CHOICE]
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ GPU Device: {device} ({torch.cuda.get_device_name(0) if device=="cuda" else "CPU"})")
print(f"📦 Đang nạp mô hình: {MODEL_CHOICE}...")

if "hub" in cfg:
    model, _, preprocess = open_clip.create_model_and_transforms(cfg["hub"])
else:
    model, _, preprocess = open_clip.create_model_and_transforms(cfg["model"], pretrained=cfg["pretrained"])

model.to(device).eval()
print(f"✅ Mô hình {MODEL_CHOICE} đã sẵn sàng!")

In [ ]:
# [Step 3] Clone Dataset FracAtlas về Colab
print("📥 Đang clone tập dữ liệu FracAtlas...")
!git clone --depth 1 https://github.com/huyen-nguyen/FracAtlas.git ./fracatlas_repo || true

image_files = list(Path("./fracatlas_repo").rglob("*.jpg")) + list(Path("./fracatlas_repo").rglob("*.png"))
print(f"📷 Tìm thấy {len(image_files)} tệp ảnh X-quang trong FracAtlas!")

In [ ]:
# [Step 4] Mã hóa Batch GPU các ảnh X-quang
vectors = []
metadata = []

print(f"🚀 Bắt đầu trích xuất Vector cho {len(image_files)} ảnh X-quang...")
for img_path in tqdm(image_files, desc="GPU Vector Extraction"):
    try:
        img = Image.open(img_path).convert("RGB")
        with torch.no_grad():
            tensor = preprocess(img).unsqueeze(0).to(device)
            feat = model.encode_image(tensor)
            feat /= feat.norm(dim=-1, keepdim=True)
            vectors.append(feat[0].cpu().numpy().astype(np.float32))

        is_frac = "fractured" in img_path.name.lower() or "fracture" in str(img_path.parent).lower()
        metadata.append({
            "image_id": f"fracatlas-{"fractured" if is_frac else "normal"}-{img_path.stem.lower()}",
            "title": f"FracAtlas X-ray {img_path.name}",
            "body_part": "forearm/wrist",
            "diagnosis": "fracture" if is_frac else "normal",
            "fracture_type": "fractured" if is_frac else "none",
            "region": "forearm and wrist",
            "evidence_note": f"FracAtlas real X-ray dataset case {img_path.name}.",
            "text": f"fracatlas {"fractured" if is_frac else "normal"} xray wrist forearm bone case {img_path.stem.lower()}",
            "image_path": str(img_path)
        })
    except Exception:
        continue

print(f"\n✅ Đã mã hóa xong {len(vectors)} vectors!")

In [ ]:
# [Step 5] Tạo FAISS Index & Xuất file lưu trữ Disk
vec_matrix = np.array(vectors, dtype=np.float32)
dim = vec_matrix.shape[1]
prefix = cfg["prefix"]

index = faiss.IndexFlatIP(dim)
faiss.normalize_L2(vec_matrix)
index.add(vec_matrix)

faiss_filename = f"{prefix}.faiss"
meta_filename = f"{prefix}_metadata.json"

faiss.write_index(index, faiss_filename)
with open(meta_filename, "w", encoding="utf-8") as fh:
    json.dump(metadata, fh, ensure_ascii=False, indent=2)

print("\n=======================================================")
print(f"🎉 HOÀN THÀNH XÂY CHỈ SỐ CHO {MODEL_CHOICE}!")
print(f"  • Kích thước Vector Dim: {dim}")
print(f"  • Số bản ghi Indexed: {index.ntotal}")
print(f"\n📁 File đã xuất:")
print(f"  1. {faiss_filename}  (FAISS Index file)")
print(f"  2. {meta_filename}   (Metadata JSON file)")
print("=======================================================")